# Semantic segmentation rendering

This notebook demonstrates the semantic segmentation rendering APIs added in this branch:

- `mjw.get_segmentation(...)`
- `mjw.get_semantic_segmentation(...)`

Unlike `tutorial.ipynb`, this notebook imports `mujoco_warp` directly from the local checkout so branch-only features are available immediately. It also disables IPython autoreload for the session, because Warp kernels need file-backed Python source.


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
  for candidate in [start.resolve(), *start.resolve().parents]:
    if (candidate / 'pyproject.toml').exists() and (candidate / 'mujoco_warp').is_dir():
      return candidate
  raise RuntimeError(
      'Run this notebook from a local mujoco_warp checkout, or open Jupyter from the repo root.'
  )

repo_root = find_repo_root(Path.cwd())
os.chdir(repo_root)

# Warp kernels need file-backed source. IPython autoreload can re-exec modules
# from strings, which breaks Warp's source extraction inside notebooks.
ip = globals().get('get_ipython', lambda: None)()
if ip is not None:
  try:
    ip.run_line_magic('autoreload', '0')
    print('Disabled IPython autoreload for this notebook.')
  except Exception:
    pass

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'mediapy', 'matplotlib'],
    check=True,
)

if any(name == 'mujoco_warp' or name.startswith('mujoco_warp.') for name in sys.modules):
  raise RuntimeError(
      'Restart the kernel and rerun this notebook from the top. '
      'mujoco_warp was imported before the local checkout path was configured.'
  )

if str(repo_root) not in sys.path:
  sys.path.insert(0, str(repo_root))
importlib.invalidate_caches()

print(f'Using local checkout: {repo_root}')


In [ ]:
import matplotlib.pyplot as plt
import mujoco
import numpy as np
import warp as wp

import mujoco_warp as mjw

wp.config.quiet = True


## Load a model with geoms and flex objects

We use `flex/multiflex.xml`, which contains a plane geom plus multiple flex objects with different dimensions. That gives us a good rendering target for both regular geom hits and `mjOBJ_FLEX` hits.


In [ ]:
xml_path = repo_root / 'mujoco_warp' / 'test_data' / 'flex' / 'multiflex.xml'
mj_model = mujoco.MjModel.from_xml_path(str(xml_path))

model = mjw.put_model(mj_model)
data = mjw.make_data(mj_model, nworld=1)
mjw.forward(model, data)

print(f'Loaded {xml_path.name}')
print(f'  geoms: {mj_model.ngeom}')
print(f'  flex objects: {mj_model.nflex}')
print(f'  cameras: {mj_model.ncam}')


## Render RGB, depth, legacy segmentation, and semantic segmentation

The legacy segmentation buffer stores geom ids for geom hits and uses `-2` as a sentinel for flex hits. The semantic segmentation buffer instead returns `(object_id, object_type)` for every hit, so flex pixels keep their real `flex_id` and are tagged as `mjOBJ_FLEX`.


In [ ]:
cam_res = (256, 256)
rc = mjw.create_render_context(
    mj_model,
    nworld=1,
    cam_res=cam_res,
    render_rgb=True,
    render_depth=True,
    render_seg=True,
)

mjw.refit_bvh(model, data, rc)
mjw.render(model, data, rc)

rgb_data = wp.zeros((1, cam_res[1], cam_res[0]), dtype=wp.vec3)
depth_data = wp.zeros((1, cam_res[1], cam_res[0]), dtype=float)
legacy_seg_data = wp.zeros((1, cam_res[1], cam_res[0]), dtype=int)
semantic_seg_data = wp.zeros((1, cam_res[1], cam_res[0], 2), dtype=int)

mjw.get_rgb(rc, camera_index=0, rgb_out=rgb_data)
mjw.get_depth(rc, camera_index=0, depth_scale=6.0, depth_out=depth_data)
mjw.get_segmentation(rc, camera_index=0, seg_out=legacy_seg_data)
mjw.get_semantic_segmentation(rc, camera_index=0, seg_out=semantic_seg_data)

rgb = rgb_data.numpy()[0]
depth = depth_data.numpy()[0]
legacy_seg = legacy_seg_data.numpy()[0]
semantic_seg = semantic_seg_data.numpy()[0]

object_id = semantic_seg[..., 0]
object_type = semantic_seg[..., 1]
geom_mask = object_type == int(mjw.ObjType.GEOM)
flex_mask = object_type == int(mjw.ObjType.FLEX)
flex_id_map = np.where(flex_mask, object_id, -1)

type_names = {
    -1: 'background',
    int(mjw.ObjType.GEOM): 'mjOBJ_GEOM',
    int(mjw.ObjType.FLEX): 'mjOBJ_FLEX',
}
visible_types = {
    type_names.get(int(value), str(int(value))): int((object_type == value).sum())
    for value in np.unique(object_type)
}
visible_flexes = {
    int(flex_id): mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_FLEX, int(flex_id))
    for flex_id in np.unique(object_id[flex_mask])
}

print('Visible object types:', visible_types)
print('Visible flex ids:', visible_flexes)
print('Legacy values on flex pixels:', np.unique(legacy_seg[flex_mask]).tolist())
print('Semantic flex ids:', np.unique(object_id[flex_mask]).tolist())
print('Geom ids match semantic object ids on geom hits:', np.array_equal(object_id[geom_mask], legacy_seg[geom_mask]))


In [ ]:
def colorize_labels(labels, invalid=-1):
  labels = labels.astype(np.int32)
  palette = np.array([
      [0.05, 0.05, 0.08],
      [0.90, 0.20, 0.20],
      [0.20, 0.75, 0.35],
      [0.20, 0.45, 0.90],
      [0.90, 0.70, 0.20],
      [0.70, 0.30, 0.85],
      [0.15, 0.75, 0.75],
      [0.95, 0.50, 0.15],
  ], dtype=np.float32)
  rgb = np.zeros(labels.shape + (3,), dtype=np.float32)
  valid = labels != invalid
  rgb[~valid] = palette[0]
  rgb[valid] = palette[(labels[valid] % (len(palette) - 1)) + 1]
  return rgb


fig, axes = plt.subplots(2, 3, figsize=(14, 10))
axes = axes.ravel()

axes[0].imshow(rgb)
axes[0].set_title('RGB')
axes[1].imshow(depth, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Depth')
axes[2].imshow(colorize_labels(legacy_seg))
axes[2].set_title('Legacy segmentation\n(geom id, flex = -2)')
axes[3].imshow(colorize_labels(object_id))
axes[3].set_title('Semantic object id')
axes[4].imshow(colorize_labels(object_type))
axes[4].set_title('Semantic object type')
axes[5].imshow(colorize_labels(flex_id_map))
axes[5].set_title('Semantic flex id')

for ax in axes:
  ax.axis('off')

plt.tight_layout()
plt.show()


## Takeaways

- `get_segmentation(...)` remains the backward-compatible geom-id buffer.
- `get_semantic_segmentation(...)` preserves both the hit id and the hit type.
- Flex hits are where the new API matters most: the legacy buffer only returns `-2`, while the semantic buffer keeps the real `flex_id` and reports `mjOBJ_FLEX`.
